# Data Quality Audit

## Load Raw Data & Verifikasi Lineage (Data Integrity)

In [7]:
import pandas as pd
import numpy as np
import hashlib
import json
import re

pd.set_option('display.max_colwidth', 100)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

meta_stage1 = json.load(open('../data/raw/collection_metadata.json'))
actual_hash = sha256('../data/raw/raw_comments_kopdes.parquet')

print('Checksum Tahap 1  :', meta_stage1['sha256_parquet'])
print('Checksum saat ini :', actual_hash)
print('LINEAGE VALID     :', actual_hash == meta_stage1['sha256_parquet'])
assert actual_hash == meta_stage1['sha256_parquet'], 'Raw data berubah sejak Tahap 1! Audit dibatalkan.'

df = pd.read_parquet('../data/raw/raw_comments_kopdes.parquet')
print()
print(f'Shape: {df.shape}')
df.head(3)

Checksum Tahap 1  : f587277b8ce852cdb65caefc1a3573b606071df24d6e0e0470a4fbc5463a3530
Checksum saat ini : f587277b8ce852cdb65caefc1a3573b606071df24d6e0e0470a4fbc5463a3530
LINEAGE VALID     : True

Shape: (220051, 8)


,video_id,comment_id,parent_comment_id,level,username,nickname,comment,create_time
0,7526458489577737488,7643302845156410130,NaN,0.0,jbm123jayabetonmandiri,Christian sudartio,Beliau bilang untung nya 2 m ?,2026-05-24 10:58:57
1,7526458489577737488,7643361778294866706,7.643303e+18,1.0,sahrulmustakimm,sahrul,Makasih Mas,2026-05-24 14:47:33
2,7526458489577737488,7643368472755487495,7.643303e+18,1.0,jbm123jayabetonmandiri,Christian sudartio,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,2026-05-24 15:13:29


## Audit Missing Value

In [8]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
audit_missing = pd.DataFrame({'n_missing': missing, 'pct_missing': missing_pct})
audit_missing

,n_missing,pct_missing
video_id,0,0.00
comment_id,0,0.00
parent_comment_id,185824,84.45
level,92912,42.22
username,0,0.00
nickname,23,0.01
comment,305,0.14
create_time,0,0.00


In [9]:
print('Interpretasi:')
print(f"- parent_comment_id kosong {missing['parent_comment_id']:,} ({missing_pct['parent_comment_id']}%) -> WAJAR, ini komentar top-level (bukan reply)")
print(f"- level kosong {missing['level']:,} ({missing_pct['level']}%) -> DIAUDIT LEBIH LANJUT di bagian 3 (ditemukan terkait duplikasi baris)")
print(f"- nickname kosong {missing['nickname']:,} ({missing_pct['nickname']}%) -> minor, akun dengan nickname kosong/privat")
print(f"- comment kosong {missing['comment']:,} ({missing_pct['comment']}%) -> tidak ada teks untuk dianalisis, kandidat drop di tahap cleaning")

Interpretasi:
- parent_comment_id kosong 185,824 (84.45%) -> WAJAR, ini komentar top-level (bukan reply)
- level kosong 92,912 (42.22%) -> DIAUDIT LEBIH LANJUT di bagian 3 (ditemukan terkait duplikasi baris)
- nickname kosong 23 (0.01%) -> minor, akun dengan nickname kosong/privat
- comment kosong 305 (0.14%) -> tidak ada teks untuk dianalisis, kandidat drop di tahap cleaning


## Audit Duplikasi Baris

In [10]:
print('Duplikasi baris penuh (semua kolom identik) :', df.duplicated().sum())
print('Duplikasi (video_id, comment_id)            :', df.duplicated(subset=['video_id','comment_id']).sum())

dup_pairs = df.duplicated(subset=['video_id','comment_id'], keep=False)
dup_df = df[dup_pairs].sort_values(['video_id','comment_id'])
print()
print('Distribusi level pada baris yang ter-duplikasi (video_id, comment_id):')
print(dup_df.groupby('level', dropna=False).size())

Duplikasi baris penuh (semua kolom identik) : 0
Duplikasi (video_id, comment_id)            : 92912

Distribusi level pada baris yang ter-duplikasi (video_id, comment_id):
level
0.0    92912
NaN    92912
dtype: int64


## Audit Duplikasi Baris

In [11]:
print('Duplikasi baris penuh (semua kolom identik) :', df.duplicated().sum())
print('Duplikasi (video_id, comment_id)            :', df.duplicated(subset=['video_id','comment_id']).sum())

dup_pairs = df.duplicated(subset=['video_id','comment_id'], keep=False)
dup_df = df[dup_pairs].sort_values(['video_id','comment_id'])
print()
print('Distribusi level pada baris yang ter-duplikasi (video_id, comment_id):')
print(dup_df.groupby('level', dropna=False).size())

Duplikasi baris penuh (semua kolom identik) : 0
Duplikasi (video_id, comment_id)            : 92912

Distribusi level pada baris yang ter-duplikasi (video_id, comment_id):
level
0.0    92912
NaN    92912
dtype: int64


In [12]:
g = dup_df.groupby(['video_id', 'comment_id'])
group_sizes = g.size()
print('Distribusi ukuran grup duplikat:')
print(group_sizes.value_counts())

check_cols = ['username', 'nickname', 'comment', 'create_time']
n_checked, n_identical = 0, 0
for name, grp in g:
    if len(grp) == 2:
        n_checked += 1
        r0, r1 = grp.iloc[0], grp.iloc[1]
        if all((r0[c] == r1[c]) or (pd.isnull(r0[c]) and pd.isnull(r1[c])) for c in check_cols):
            n_identical += 1

print(f'\nGrup duplikat diperiksa: {n_checked:,}')
print(f'Identik di semua kolom lain (username, nickname, comment, create_time): {n_identical:,} ({n_identical/n_checked*100:.2f}%)')

Distribusi ukuran grup duplikat:
2    92912
Name: count, dtype: int64

Grup duplikat diperiksa: 92,912
Identik di semua kolom lain (username, nickname, comment, create_time): 92,912 (100.00%)


## Audit Konsistensi Logis `level` vs `parent_comment_id`

In [13]:
print('Distribusi nilai level:')
print(df['level'].value_counts(dropna=False))
print()

replies = df[df['level'] == 1.0]
print('Reply (level=1.0) dengan parent_comment_id kosong:', replies['parent_comment_id'].isnull().sum(), '/', len(replies))
print('-> Semua reply memiliki parent_comment_id valid: konsisten secara struktural.')

Distribusi nilai level:
level
0.0    92912
NaN    92912
1.0    34227
Name: count, dtype: int64

Reply (level=1.0) dengan parent_comment_id kosong: 0 / 34227
-> Semua reply memiliki parent_comment_id valid: konsisten secara struktural.


## Audit Validitas Timestamp

In [14]:
now = pd.Timestamp.now()
future = df['create_time'] > now
print('Timestamp di masa depan (data tidak valid):', future.sum())
print('Rentang waktu data:', df['create_time'].min(), '->', df['create_time'].max())
print('-> Rentang waktu realistis (10 bulan terakhir), tidak ada anomali tanggal.')

Timestamp di masa depan (data tidak valid): 0
Rentang waktu data: 2025-07-13 14:06:39 -> 2026-05-27 06:45:05
-> Rentang waktu realistis (10 bulan terakhir), tidak ada anomali tanggal.


## Audit Kualitas Teks Komentar

In [15]:
com = df['comment'].dropna().astype(object).astype(str)

lengths = com.str.len()
print('=== Distribusi panjang komentar (karakter) ===')
print(lengths.describe())
print()
print('Komentar dengan panjang <=1 char :', (lengths <= 1).sum())
print('Komentar dengan panjang <=2 char :', (lengths <= 2).sum())
print('Komentar dengan panjang >500 char:', (lengths > 500).sum())

=== Distribusi panjang komentar (karakter) ===
count    219746.000000
mean         52.709328
std          61.956748
min           1.000000
25%          20.000000
50%          35.000000
75%          63.000000
max        2200.000000
Name: comment, dtype: float64

Komentar dengan panjang <=1 char : 1480
Komentar dengan panjang <=2 char : 2502
Komentar dengan panjang >500 char: 403


In [16]:
emoji_pattern = re.compile(
    "[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]+",
    flags=re.UNICODE
)

def is_emoji_only(s):
    stripped = emoji_pattern.sub('', s).strip()
    return len(stripped) == 0 and len(s.strip()) > 0

def is_punct_only(s):
    return bool(re.fullmatch(r'[\W_]+', s.strip())) if s.strip() else False

n_emoji_only = com.apply(is_emoji_only).sum()
n_punct_only = com.apply(is_punct_only).sum()
n_sticker = (com.str.strip() == '[Sticker]').sum()
n_url = com.str.contains(r'https?://|www\.', regex=True, na=False).sum()
n_repeated_char = com.apply(lambda s: bool(re.search(r'(.)\1{6,}', s))).sum()

print(f'Komentar hanya emoji                 : {n_emoji_only:,}')
print(f'Komentar hanya tanda baca             : {n_punct_only:,}')
print(f'Komentar placeholder "[Sticker]" murni : {n_sticker:,}')
print(f'Komentar mengandung URL               : {n_url:,}')
print(f'Komentar dgn karakter berulang >=7x    : {n_repeated_char:,} (perlu review manual: sebagian adalah ekspresi wajar spt "wkwkwkwk", bukan spam)')

Komentar hanya emoji                 : 6,736
Komentar hanya tanda baca             : 6,975
Komentar placeholder "[Sticker]" murni : 2,939
Komentar mengandung URL               : 41
Komentar dgn karakter berulang >=7x    : 1,533 (perlu review manual: sebagian adalah ekspresi wajar spt "wkwkwkwk", bukan spam)


## Audit Potensi Bot / Akun Spam

In [17]:
top_users = df['username'].value_counts()
print('Jumlah akun unik           :', df['username'].nunique())
print('Akun dengan >100 komentar  :', (top_users > 100).sum())
print()
print('Top 10 akun paling aktif:')
print(top_users.head(10))
print()
print('-> Aktivitas tersebar (akun paling aktif hanya 191 komentar dari 220rb total),')
print('   tidak ada indikasi kuat serangan bot/buzzer terkoordinasi berskala besar.')

Jumlah akun unik           : 102899
Akun dengan >100 komentar  : 1

Top 10 akun paling aktif:
username
u_mey24                  191
astotoedysalem            45
koperasimerahputih103     44
sehatitumahalya3          43
lamongankidul3            42
ardyar.ayi.rosadi         38
dndsm_                    34
mind_hanna                33
restu.erlangga0           32
jasminefortuna11          29
Name: count, dtype: int64

-> Aktivitas tersebar (akun paling aktif hanya 191 komentar dari 220rb total),
   tidak ada indikasi kuat serangan bot/buzzer terkoordinasi berskala besar.


## Audit Skrip Non-Latin (Potensi Bahasa Non-Indonesia)

In [18]:
non_latin_mask = com.apply(lambda s: bool(re.search(r'[\u0600-\u06FF\u4e00-\u9fff\u3040-\u30ff]', s)))
print('Baris mengandung karakter Arab/CJK/Jepang:', non_latin_mask.sum(), f'({non_latin_mask.sum()/len(com)*100:.3f}%)')
print(com[non_latin_mask].head(5).tolist())
print()
print('-> Proporsi sangat kecil (<0.02%), umumnya berupa mention username asing (@nama).')
print('   Tidak signifikan mempengaruhi kualitas dataset secara keseluruhan.')

Baris mengandung karakter Arab/CJK/Jepang: 36 (0.016%)
['@アリン', '@热罕 😭😭', 'GAKUAT BGT 😭 @اوى @inichikiiy', '@ディアナ @trynn @teletubiesnaiksemut anjjj😭', '@🧘🏻\u200d♀️ @🫐سلوى']

-> Proporsi sangat kecil (<0.02%), umumnya berupa mention username asing (@nama).
   Tidak signifikan mempengaruhi kualitas dataset secara keseluruhan.


## Audit Per-Video (Distribusi Volume & Kualitas)

In [19]:
com_full = df['comment'].astype(object)
per_video = df.assign(comment_len=com_full.str.len()).groupby('video_id').agg(
    n_comments=('comment_id', 'size'),
    n_missing_comment=('comment', lambda s: s.isnull().sum()),
    avg_comment_len=('comment_len', 'mean')
).sort_values('n_comments', ascending=False)

per_video['avg_comment_len'] = per_video['avg_comment_len'].round(1)
per_video

,n_comments,n_missing_comment,avg_comment_len
video_id,,,
7641276206687554823,26737,9,61.7
7641470196108004626,20717,17,55.1
7643478603576954132,19421,22,63.5
7641041003423501576,19070,25,36.8
7640891074545749266,14106,22,58.5
7641183642437586196,12874,20,54.7
7637357539209841941,12250,24,43.1
7640835929623579912,11409,25,44.9
7640152776986660103,10856,28,29.4


## Ringkasan Audit & Rekomendasi untuk Tahap Cleaning

| # | Temuan | Jumlah | % dari Total | Tindakan Direkomendasikan (Tahap 4: Cleaning) |
|---|---|---|---|---|
| 1 | Duplikat scraper (video_id+comment_id, konten identik) | 92.912 | 42,22% | **Drop** duplikat, simpan baris dengan `level` terisi |
| 2 | Comment kosong (NaN) | 305 | 0,14% | **Drop** (tidak ada teks untuk dianalisis) |
| 3 | Komentar placeholder `[Sticker]` murni | 2.939 | 1,34% | **Drop** atau beri label terpisah (bukan teks natural) |
| 4 | Komentar hanya emoji | 6.736 | 3,06% | **Kaji ulang**: sebagian bisa dipetakan ke sentimen dari emoji, sebagian di-drop jika tanpa konteks |
| 5 | Komentar hanya tanda baca | 6.975 | 3,17% | **Drop** (tidak mengandung sinyal tekstual) |
| 6 | Komentar sangat pendek (<=2 char) | 2.502 | 1,14% | **Kaji ulang** manual (mis. "ya", "no") |
| 7 | Komentar mengandung URL | 41 | 0,02% | **Bersihkan** URL saat preprocessing (remove URL), tidak perlu drop baris |
| 8 | nickname kosong | 23 | 0,01% | **Biarkan** (kolom tidak dipakai untuk sentiment) |
| 9 | Karakter non-Latin (Arab/CJK) | 36 | 0,02% | **Biarkan**, dampak dapat diabaikan |
| 10 | Timestamp tidak valid / masa depan | 0 | 0% | Tidak ada tindakan |

## Simpan Hasil Audit

In [20]:
import os
os.makedirs('reports', exist_ok=True)

audit_summary = {
    'lineage_verified': True,
    'n_rows_raw': int(len(df)),
    'n_duplicate_scraper_rows': 92912,
    'n_comment_null': 305,
    'n_sticker_only': int(n_sticker),
    'n_emoji_only': int(n_emoji_only),
    'n_punct_only': int(n_punct_only),
    'n_very_short_le2': int((lengths <= 2).sum()),
    'n_contains_url': int(n_url),
    'n_nickname_null': 23,
    'n_non_latin_script': int(non_latin_mask.sum()),
    'n_future_timestamps': int(future.sum()),
    'estimated_clean_rows_min': 120000,
    'estimated_clean_rows_max': 130000,
}

with open('../reports/data_quality_audit_summary.json', 'w') as f:
    json.dump(audit_summary, f, indent=2)

print(json.dumps(audit_summary, indent=2))

{
  "lineage_verified": true,
  "n_rows_raw": 220051,
  "n_duplicate_scraper_rows": 92912,
  "n_comment_null": 305,
  "n_sticker_only": 2939,
  "n_emoji_only": 6736,
  "n_punct_only": 6975,
  "n_very_short_le2": 2502,
  "n_contains_url": 41,
  "n_nickname_null": 23,
  "n_non_latin_script": 36,
  "n_future_timestamps": 0,
  "estimated_clean_rows_min": 120000,
  "estimated_clean_rows_max": 130000
}
